# Text Classification with Scikit-Learn

The goal of this workshop is to explore text classification with scikit-learn, a widely used library.

In this pre-work, you will see how to do text classification on a collection of text documents on different topics.

We will:
- Read in data
- Extract feature vectors
- Train a linear model to perform categorization
- Find a good configuration of both the feature extraction components and the classifier

These activities are adapted from the "Working with Text Data" scikit-learn tutorial (no longer available, but contained in [v1.4](https://scikit-learn.org/1.4/tutorial/text_analytics/working_with_text_data.html) of the documentation).

**You are welcome to download and run this notebook somewhere else** (e.g., on your local computer). If you do, you will need to have [scikit-learn](https://scikit-learn.org/1.4/install.html#installation-instructions) installed.


## Loading the 20 newsgroups dataset

We will be using a classic dataset called “Twenty Newsgroups”, "a collection of approximately 20,000 newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups." [official website](http://qwone.com/~jason/20Newsgroups/).

The code below uses scikit-learn's built-in dataset loader for 20 newsgroups. Alternatively, it is possible to download the dataset manually from the website and use the [sklearn.datasets.load_files](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_files.html) function by pointing it to the `20news-bydate-train sub-folder` of the uncompressed archive folder.

In order to get faster execution times for this first example, we will work on a partial dataset with only 4 categories out of the 20 available in the dataset:

In [88]:
categories = ['comp.sys.mac.hardware', 'rec.sport.baseball',
               'comp.graphics', 'sci.med']

We can now load the list of files matching those categories as follows:

In [89]:
from sklearn.datasets import fetch_20newsgroups

twenty_train = fetch_20newsgroups(subset='train',
     categories=categories, shuffle=True, random_state=42)

The returned dataset is a scikit-learn “bunch”, an object that lets you access fields like a dictionary or an object. For example, the `target_names` holds the list of the requested category names (only four of the twenty because these are the ones we said should be fetched above):

In [90]:
twenty_train.target_names

['comp.graphics', 'comp.sys.mac.hardware', 'rec.sport.baseball', 'sci.med']

The files themselves are loaded in memory in the `data` attribute:

In [91]:
print("Data", len(twenty_train.data))

Data 2353


We can also get the filenames:

In [92]:
print("Filenames", len(twenty_train.filenames))

Filenames 2353


Let’s print the first lines of the first loaded file and its label:

In [93]:
print("Label:", twenty_train.target_names[twenty_train.target[0]])
print("\n".join(twenty_train.data[0].split("\n")[:3]))

Label: sci.med
From: robin@ntmtv.com (Robin Coutellier)
Subject: Critique of Pressure Point Massager
Originator: robin@volans


For our model, the news cateogry will be the label.

For speed and space efficiency reasons, scikit-learn loads the target attribute as an array of integers that corresponds to the index of the category name in the `target_names` list. The category integer id of each sample is stored in the `target` attribute:

In [94]:
twenty_train.target[:10]

array([3, 0, 1, 0, 2, 0, 0, 1, 3, 2])

It is possible to get back the category names as follows:

In [95]:
for t in twenty_train.target[:10]:
    print(twenty_train.target_names[t])

sci.med
comp.graphics
comp.sys.mac.hardware
comp.graphics
rec.sport.baseball
comp.graphics
comp.graphics
comp.sys.mac.hardware
sci.med
rec.sport.baseball


## Extracting features from text files

In order to perform machine learning on text documents, we first need to turn the text content into a numerical representation (as discussed in lecture 1 and 2).

### Bag of words

We will start with the approach described in lecture 1, a bag of words:

1. Assign an ID to each word in any document of the training set.
2. For each document `d` and each word `w`, count how often `w` appears in `d` and store the value in `X[d, w]`.

We will represent this with `scipy.sparse` matrices. In assignment 1, we don't let you use these libraries, so you can develop an intuition for how they work. In practise, you should not use your own implementation as other people have spent a lot of time optimising these implementations.

### Tokenising text with scikit-learn


To count tokens we can use [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#). As well as counting, it can also perfom a range of text processing steps, including tokenising text, filtering stopwords, ignoring very frequent or very rare features, and so on. It produces an object that can then transform a document into a feature vector.

In [96]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(twenty_train.data)
X_train_counts.shape

(2353, 33868)

[CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#) supports counts of N-grams of words or characters.

Once fitted, the vectorizer has built a dictionary of feature indices:

In [97]:
count_vect.vocabulary_.get(u'algorithm')

5071

The index value of a word in the vocabulary is linked to its frequency in the whole training corpus.

### TF-IDF

As discussed in lecture 1, raw frequencies are often not quite what we want. We described the TF-IDF equation, which changes the value we store for a word, by (1) squashing the frequency, and (2) rescaling it depending on how many different documents the word appears in.

TF-IDF can be computed using [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html):


In [98]:
from sklearn.feature_extraction.text import TfidfTransformer
tf_transformer = TfidfTransformer(use_idf=False).fit(X_train_counts)
X_train_tf = tf_transformer.transform(X_train_counts)
X_train_tf.shape

(2353, 33868)

In the code above, we first use the `fit(..)` method to get counts from the data and then use the `transform(..)` method to transform our count-matrix to a tf-idf representation.

These two steps can be combined to achieve the same end result faster by using the `fit_transform(..)` method:

In [99]:
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_train_tfidf.shape

(2353, 33868)

## Training a classifier

Now that we have our features, we can train a classifier to try to predict the category of a post. Let’s start with a [naïve Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html#naive-bayes) classifier, which provides a nice baseline for this task. Note, naïve Bayeswas briefly mentioned in lecture 2, but you are not expected to understand the details of the method.

scikit-learn includes several variants of naïve Bayes, and the one most suitable for word counts is the multinomial variant:


In [100]:
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)

To try to predict the outcome on a new document we need to extract the features using almost the same feature extracting steps as before. The difference is that we call `transform` instead of `fit_transform` on the transformers. `transform` applies the change to new data. `fit_transform` fits the parameters of the transformer:

In [101]:
docs_new = ['God is love', 'OpenGL on the GPU is fast']
X_new_counts = count_vect.transform(docs_new)
X_new_tfidf = tfidf_transformer.transform(X_new_counts)

predicted = clf.predict(X_new_tfidf)

for doc, category in zip(docs_new, predicted):
    print('%r => %s' % (doc, twenty_train.target_names[category]))

'God is love' => rec.sport.baseball
'OpenGL on the GPU is fast' => comp.graphics


## Building a pipeline

In order to make the vectorizer => transformer => classifier easier to work with, scikit-learn provides a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) class that behaves like a classifier, doing all of the steps together, passing the output of one as the input to the next:


In [102]:
from sklearn.pipeline import Pipeline
text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

The names `vect`, `tfidf` and `clf` (classifier) are arbitrary. We will use them to perform grid search for suitable hyperparameters below. We can now train the model with a single command:



In [103]:
text_clf.fit(twenty_train.data, twenty_train.target)

Pipeline(steps=[('vect', CountVectorizer()), ('tfidf', TfidfTransformer()),
                ('clf', MultinomialNB())])

## Evaluation of the performance on the test set

Now we will get the test data, which the model has not seen, and use it to measure how accurate the model is:

In [104]:
import numpy as np
twenty_test = fetch_20newsgroups(subset='test',
    categories=categories, shuffle=True, random_state=42)
docs_test = twenty_test.data
predicted = text_clf.predict(docs_test)
np.mean(predicted == twenty_test.target)

np.float64(0.9329929802169751)

We achieved 93.3% accuracy. Let’s see if we can do better with a linear [support vector machine (SVM)](https://scikit-learn.org/stable/modules/svm.html#svm), which is widely regarded as one of the best linear text classification algorithms. We can change the learner by simply plugging a different classifier object into our pipeline:

In [105]:
from sklearn.linear_model import SGDClassifier
text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(loss='hinge', penalty='l2',
                          alpha=1e-3, random_state=42,
                          max_iter=5, tol=None)),
])

text_clf.fit(twenty_train.data, twenty_train.target)
predicted = text_clf.predict(docs_test)
np.mean(predicted == twenty_test.target)

np.float64(0.9432035737077218)

We achieved 94.3% accuracy using the SVM. scikit-learn provides further utilities for more detailed performance analysis of the results:

In [106]:
from sklearn import metrics
print(metrics.classification_report(twenty_test.target, predicted,
    target_names=twenty_test.target_names))

                       precision    recall  f1-score   support

        comp.graphics       0.92      0.92      0.92       389
comp.sys.mac.hardware       0.95      0.95      0.95       385
   rec.sport.baseball       0.94      0.99      0.97       397
              sci.med       0.97      0.92      0.94       396

             accuracy                           0.94      1567
            macro avg       0.94      0.94      0.94      1567
         weighted avg       0.94      0.94      0.94      1567



Most of these metrics are as described in lecture 2.

We can see that we are doing slightly better on baseball.

Now let's look at the confusion matrix

In [107]:
metrics.confusion_matrix(twenty_test.target, predicted)

array([[358,  12,  10,   9],
       [ 10, 364,   8,   3],
       [  0,   3, 393,   1],
       [ 22,   5,   6, 363]])

Baseball is very different from medicine and computer related documents, so it is rarely confused for them (see the third row and column). The two computer related ones are easy to confuse.

## Hyperparameter tuning using grid search

Note: this section may not work on Ed due to resource constraints.

We’ve already encountered some hyperparameters such as `use_idf` in the `TfidfTransformer`. Classifiers tend to have many hyperparameters as well; e.g., `MultinomialNB` includes a smoothing hyperparameter `alpha`, and `SGDClassifier` has a penalty hyperparameter `alpha` and configurable loss and penalty terms in the objective function (see the module documentation, or use the Python `help(...)` function to get a description of these).

We could manually try various settings of these hyperparameters to find the configuration that leads to the best results. That would be a lot of effort though. Instead, we can automatically search through a range of combinations of options. The simplest search is a grid search. 

Let's try out using (a) either words or bigrams, (b) with or without idf, and (c) with a penalty parameter of either 0.01 or 0.001 for the linear SVM:



In [108]:
from sklearn.model_selection import GridSearchCV
parameters = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'tfidf__use_idf': (True, False),
    'clf__alpha': (1e-2, 1e-3),
}

The more options you consider, the more effort this search will involve. In this case we are considering 2x2x2 = 8 combinations.

If we have multiple CPU cores at our disposal, we can tell the grid searcher to try these eight parameter combinations in parallel by using the `n_jobs` parameter. If we give this parameter a value of `-1`, grid search will detect how many cores are installed and use them all.

When running this in Ed, we use just 1 core, but you can use more on your own machine.

In [109]:
gs_clf = GridSearchCV(text_clf, parameters, cv=5, n_jobs=1)

The grid search instance behaves like a normal scikit-learn model. Let’s perform the search on a smaller subset of the training data to speed up the computation:

In [110]:
gs_clf = gs_clf.fit(twenty_train.data[:400], twenty_train.target[:400])

The result of calling `fit` on a `GridSearchCV` object is a classifier that we can use to run `predict` to get a guess at the answer for an input:

In [111]:
twenty_train.target_names[gs_clf.predict(['Computers are cool'])[0]]

'comp.sys.mac.hardware'

The object’s `best_score_` and `best_params_` attributes store the best mean score and the parameters setting corresponding to that score:

In [112]:
gs_clf.best_score_
for param_name in sorted(parameters.keys()):
    print("%s: %r" % (param_name, gs_clf.best_params_[param_name]))

clf__alpha: 0.001
tfidf__use_idf: True
vect__ngram_range: (1, 1)


A more detailed summary of the search is available at `gs_clf.cv_results_`.

The `cv_results_` parameter can be imported into pandas as a DataFrame for further inspection.

You're done with the pre-work part of this workshop!